# Query Generation Notebook

This notebook implements a query generation pipeline following these steps:
1. Choose a topic from the 24 topic categories
2. Load the relevant CSV file with topic-specific documents
3. Select top-K documents by confidence score
4. Pick a random target document and get relevant documents
5. Search for documents in parquet shards to get full content
6. Generate embeddings using BGE-M3 model
7. Create FAISS index and find similar documents
8. Extract keywords and compute similarities
9. Generate query using LLM with keyword clues

In [5]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np
import random
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# For embeddings
from sentence_transformers import SentenceTransformer
import torch

# For FAISS
import faiss

# For keyword extraction
from keybert import KeyBERT
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# For LLM
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("All imports completed successfully!")

All imports completed successfully!


In [6]:
# CUDA Check Cell
print("=" * 50)
print("CUDA STATUS CHECK")
print("=" * 50)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")
print("=" * 50)

CUDA STATUS CHECK
PyTorch version: 2.7.1+cu126
CUDA available: True
GPU: NVIDIA RTX A4500
Memory: 19.6 GB
Using device: cuda


In [33]:
# Cell 2: Configuration and Constants

# Topic selection (from the 24 categories)
TOPIC = "entertainment"  # You can change this to any of the 24 topics

# File paths
TOPIC_CSV_PATH = f"/workspace/topic_grouped_csv/{TOPIC}.csv"
PARQUET_SHARDS_DIR = "/workspace/split_parquet_shards/"

# Parameters
TOP_K = 100  # Number of top documents to consider
FAISS_TOP_K = 10  # Number of similar documents to retrieve from FAISS
KEYWORD_TOP_K = 20  # Number of keywords to extract

# Model configurations
EMBEDDING_MODEL = "BAAI/bge-m3"  # Can be changed to other models
KEYWORD_MODEL = "all-MiniLM-L6-v2"  # For KeyBERT
LLM_MODEL = "HuggingFaceH4/zephyr-7b-beta"  # Can be changed to other models

print(f"Configuration:")
print(f"Topic: {TOPIC}")
print(f"Top-K: {TOP_K}")
print(f"Embedding Model: {EMBEDDING_MODEL}")
print(f"LLM Model: {LLM_MODEL}")

Configuration:
Topic: entertainment
Top-K: 100
Embedding Model: BAAI/bge-m3
LLM Model: HuggingFaceH4/zephyr-7b-beta


In [8]:
# Cell 3: Load Topic-Specific CSV and Select Top-K Documents

def load_topic_csv(csv_path, top_k=100):
    """Load topic-specific CSV and return top-K documents by confidence."""
    print(f"Loading CSV from: {csv_path}")
    
    # Load the CSV file
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} documents from {TOPIC} topic")
    print(f"Columns: {df.columns.tolist()}")
    
    # Sort by confidence score and get top-K
    df_sorted = df.sort_values('confidence', ascending=False).head(top_k)
    print(f"Selected top {len(df_sorted)} documents by confidence")
    print(f"Confidence range: {df_sorted['confidence'].min():.4f} - {df_sorted['confidence'].max():.4f}")
    
    return df_sorted

# Load the topic CSV
topic_df = load_topic_csv(TOPIC_CSV_PATH, TOP_K)
print("\nSample of top documents:")
print(topic_df.head())

Loading CSV from: /workspace/topic_grouped_csv/entertainment.csv
Loaded 830263 documents from entertainment topic
Columns: ['id', 'title', 'confidence']
Selected top 100 documents by confidence
Confidence range: 0.9999 - 0.9999

Sample of top documents:
              id                          title  confidence
574322  55597792           Here in Times Square      0.9999
655695  65682571  American Music Awards of 2020      0.9999
446807  41104517                      DJ Khushi      0.9999
385234  35296635                    Audio-Files      0.9999
219916  20366350         The Bridge (Sirius XM)      0.9999


In [9]:
# Cell 4: Select Target Document and Relevant Documents

def select_target_and_relevant_docs(df, target_idx=None):
    """Select a random target document and return relevant documents."""
    if target_idx is None:
        target_idx = random.randint(0, len(df) - 1)
    
    target_doc = df.iloc[target_idx]
    relevant_docs = df.drop(df.index[target_idx])
    
    print(f"Selected target document at index {target_idx}:")
    print(f"  ID: {target_doc['id']}")
    print(f"  Title: {target_doc['title']}")
    print(f"  Confidence: {target_doc['confidence']:.4f}")
    print(f"\nRelevant documents count: {len(relevant_docs)}")
    
    return target_doc, relevant_docs

# Select target and relevant documents
target_doc, relevant_docs = select_target_and_relevant_docs(topic_df)
target_id = target_doc['id']
relevant_ids = relevant_docs['id'].tolist()

Selected target document at index 73:
  ID: 54528313
  Title: Michael Jackson's Halloween
  Confidence: 0.9999

Relevant documents count: 99


In [10]:
# Cell 5: Search for Documents in Parquet Shards

def find_document_in_shards(doc_id, shards_dir):
    """Search for a document ID across all parquet shards."""
    shard_files = [f for f in os.listdir(shards_dir) if f.endswith('.parquet')]
    shard_files.sort()
    
    for shard_file in shard_files:
        shard_path = os.path.join(shards_dir, shard_file)
        try:
            df = pd.read_parquet(shard_path)
            if str(doc_id) in df['id'].values:
                doc = df[df['id'] == str(doc_id)].iloc[0]
                print(f"Found document {doc_id} in {shard_file}")
                return doc
        except Exception as e:
            print(f"Error reading {shard_file}: {e}")
            continue
    
    print(f"Document {doc_id} not found in any shard")
    return None

def get_documents_content(doc_ids, shards_dir):
    """Get full content for multiple document IDs."""
    documents = {}
    
    # lets search for any of the doc_ids in each of the shards and once found, add it to the documents dictionary
    # we should make only one search for each shard
    shard_files = [f for f in os.listdir(shards_dir) if f.endswith('.parquet')]
    shard_files.sort()
    for shard_file in shard_files:
        shard_path = os.path.join(shards_dir, shard_file)
        df = pd.read_parquet(shard_path)
        for doc_id in doc_ids:
            if str(doc_id) in df['id'].values:
                print(f"Found {doc_id} in {shard_file}")
                # remove the doc_id from the list
                doc_ids.remove(doc_id)
                doc = df[df['id'] == str(doc_id)].iloc[0]
                documents[str(doc_id)] = {
                    'id': doc['id'],
                    'title': doc['title'],
                    'text': doc['text'],
                    'url': doc['url']
                }
    
    return documents

# Get target document content
print("Searching for target document...")
target_content = find_document_in_shards(target_id, PARQUET_SHARDS_DIR)

if target_content is not None:
    print(f"\nTarget document content:")
    print(f"Title: {target_content['title']}")
    print(f"Text preview: {target_content['text'][:200]}...")
else:
    print("Target document not found!")
    exit()

# Get relevant documents content
print(f"\nSearching for {len(relevant_ids)} relevant documents...")
relevant_docs_content = get_documents_content(relevant_ids, PARQUET_SHARDS_DIR)
print(f"Found {len(relevant_docs_content)} relevant documents")

Searching for target document...
Found document 54528313 in shard_0044_100000docs.parquet

Target document content:
Title: Michael Jackson's Halloween
Text preview: Michael Jackson's Halloween is a one-hour animated television special that premiered on CBS on October 27, 2017. It was produced by Splash Entertainment.

The cast of voice actors includes Lucas Till,...

Searching for 99 relevant documents...
Found 4286706 in shard_0001_100000docs.parquet
Found 4275481 in shard_0001_100000docs.parquet
Found 5088064 in shard_0002_100000docs.parquet
Found 5304375 in shard_0002_100000docs.parquet
Found 4961776 in shard_0002_100000docs.parquet
Found 5250660 in shard_0002_100000docs.parquet
Found 6310226 in shard_0003_100000docs.parquet
Found 6550548 in shard_0004_100000docs.parquet
Found 11924561 in shard_0008_100000docs.parquet
Found 10294516 in shard_0008_100000docs.parquet
Found 12666624 in shard_0009_100000docs.parquet
Found 17439018 in shard_0014_100000docs.parquet
Found 17417568 in shard

In [11]:
# Cell 6: Generate Embeddings

def load_embedding_model(model_name):
    """Load the embedding model."""
    print(f"Loading embedding model: {model_name}")
    model = SentenceTransformer(model_name)
    print(f"Model loaded successfully!")
    return model

def generate_embeddings(texts, model):
    """Generate embeddings for a list of texts."""
    print(f"Generating embeddings for {len(texts)} texts...")
    embeddings = model.encode(texts, show_progress_bar=True, device=device, batch_size=8)
    print(f"Generated embeddings shape: {embeddings.shape}")
    return embeddings

# Load embedding model
embedding_model = load_embedding_model(EMBEDDING_MODEL)

# Prepare texts for embedding
target_text = target_content['text']
relevant_texts = [doc['text'] for doc in relevant_docs_content.values()]

# check if cuda is available
if torch.cuda.is_available():
    print("CUDA is available")
    device = torch.device("cuda")
else:
    print("CUDA is not available")
    device = torch.device("cpu")

# Generate embeddings
target_embedding = generate_embeddings([target_text], embedding_model)[0]
relevant_embeddings = generate_embeddings(relevant_texts, embedding_model)

print(f"\nTarget embedding shape: {target_embedding.shape}")
print(f"Relevant embeddings shape: {relevant_embeddings.shape}")

Loading embedding model: BAAI/bge-m3


Model loaded successfully!
CUDA is available
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings shape: (1, 1024)
Generating embeddings for 97 texts...


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Generated embeddings shape: (97, 1024)

Target embedding shape: (1024,)
Relevant embeddings shape: (97, 1024)


In [12]:
print("Hello World")

Hello World


In [13]:
# Cell 7: Create FAISS Index and Find Similar Documents

def create_faiss_index(embeddings, dimension):
    """Create a FAISS index for the embeddings."""
    print(f"Creating FAISS index with dimension {dimension}...")
    
    # Normalize embeddings for cosine similarity
    faiss.normalize_L2(embeddings)
    
    # Create index
    index = faiss.IndexFlatIP(dimension)  # Inner product for cosine similarity
    index.add(embeddings.astype('float32'))
    
    print(f"FAISS index created with {index.ntotal} vectors")
    return index

def search_similar_documents(query_embedding, index, relevant_docs, top_k=10):
    """Search for similar documents using FAISS."""
    print(f"Searching for top-{top_k} similar documents...")
    
    # Normalize query embedding
    query_embedding_norm = query_embedding.copy()
    faiss.normalize_L2(query_embedding_norm.reshape(1, -1))
    
    # Search
    scores, indices = index.search(query_embedding_norm.astype('float32').reshape(1, -1), top_k)
    
    # Get the relevant document IDs
    relevant_ids = list(relevant_docs_content.keys())
    
    similar_docs = []
    for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
        if idx < len(relevant_ids):
            doc_id = relevant_ids[idx]
            doc_info = relevant_docs_content[doc_id]
            similar_docs.append({
                'rank': i + 1,
                'doc_id': doc_id,
                'title': doc_info['title'],
                'score': float(score),
                'text': doc_info['text']
            })
    
    return similar_docs

# Create FAISS index
embedding_dim = relevant_embeddings.shape[1]
faiss_index = create_faiss_index(relevant_embeddings, embedding_dim)

# Search for similar documents
similar_docs = search_similar_documents(target_embedding, faiss_index, relevant_docs_content, FAISS_TOP_K)

print(f"\nTop {FAISS_TOP_K} similar documents:")
for doc in similar_docs:
    print(f"{doc['rank']}. {doc['title']} (Score: {doc['score']:.4f})")

Creating FAISS index with dimension 1024...
FAISS index created with 97 vectors
Searching for top-10 similar documents...

Top 10 similar documents:
1. Hollywood Babble-On (Score: 0.6061)
2. 2017 MTV Video Music Awards (Score: 0.5836)
3. E! True Hollywood Story (Score: 0.5738)
4. Here in Times Square (Score: 0.5654)
5. A Colbert Christmas: The Greatest Gift of All! (Score: 0.5581)
6. Studio 2054 (Score: 0.5479)
7. Greg James (radio show) (Score: 0.5445)
8. 2009 MuchMusic Video Awards (Score: 0.5362)
9. The MJCast (Score: 0.5353)
10. Sarah Cooper: Everything's Fine (Score: 0.5271)


In [14]:
# Cell 8: Extract Keywords and Compute Similarities

def load_keyword_model(model_name):
    """Load the keyword extraction model."""
    print(f"Loading keyword model: {model_name}")
    model_name = SentenceTransformer(model_name, device="cuda")
    model = KeyBERT(model_name)
    print(f"Keyword model loaded successfully!")
    return model

def extract_keywords(text, model, top_k=20):
    """Extract keywords from text using KeyBERT."""
    try:
        keywords = model.extract_keywords(text[:512],                       # limit to 512 characters for trying to avoid the error
                                         keyphrase_ngram_range=(1, 1),      # only extract single words (unigrams) for trying
                                         stop_words='english', 
                                         use_maxsum=True, 
                                         nr_candidates=10,         #top_k*2, 
                                         top_n=5) #     #top_k)
        return [kw for kw, score in keywords]
    except Exception as e:
        print(f"Error extracting keywords: {e}")
        return []

def compute_keyword_similarity(target_keywords, relevant_keywords_list):
    """Compute similarity between target keywords and relevant documents keywords."""
    # Combine all relevant keywords
    all_relevant_keywords = set()
    for keywords in relevant_keywords_list:
        all_relevant_keywords.update(keywords)
    
    # Find unique terms from target that are not in relevant
    target_keywords_set = set(target_keywords)
    unique_terms = target_keywords_set - all_relevant_keywords
    
    # If no unique terms, use top target keywords
    if not unique_terms:
        unique_terms = set(target_keywords[:10])
    
    return list(unique_terms)

# Load keyword model
keyword_model = load_keyword_model(KEYWORD_MODEL)

# Extract keywords from target document
print("Extracting keywords from target document...")
target_keywords = extract_keywords(target_content['text'], keyword_model, KEYWORD_TOP_K)
print(f"Target keywords: {target_keywords}")

# Extract keywords from similar documents
print(f"\nExtracting keywords from {len(similar_docs)} similar documents...")
similar_docs_keywords = []
for doc in similar_docs:
    keywords = extract_keywords(doc['text'], keyword_model, KEYWORD_TOP_K)
    similar_docs_keywords.append(keywords)
    print(f"Document {doc['rank']}: {keywords[:5]}...")

# Compute unique terms
unique_terms = compute_keyword_similarity(target_keywords, similar_docs_keywords)
print(f"\nUnique terms from target: {unique_terms}")

Loading keyword model: all-MiniLM-L6-v2


Keyword model loaded successfully!
Extracting keywords from target document...
Target keywords: ['animated', 'premiered', 'lucas', 'jackson', 'halloween']

Extracting keywords from 10 similar documents...
Document 1: ['hollywood', 'kroq', 'babble', 'garman', 'podcast']...
Document 2: ['broadcast', 'videos', '2017', 'katy', 'lil']...
Document 3: ['trends', 'executives', 'journalists', 'story', 'documentary']...
Document 4: ['keys', 'times', 'alicia', 'square', 'concert']...
Document 5: ['november', 'elvis', 'cabin', 'gift', 'colbert']...
Document 6: ['2054', 'angèle', 'madonna', 'dua', 'livestream']...
Document 7: ['weekday', 'interviews', 'entertainment', '00pm', 'greg']...
Document 8: ['won', '2009', 'nickelback', 'jonas', 'muchmusic']...
Document 9: ['spotify', 'entertainer', 'jamon', 'jackson', 'podcasts']...
Document 10: ['natasha', 'scooter', 'comedies', 'special', 'cooper']...

Unique terms from target: ['lucas', 'halloween', 'animated', 'premiered']


In [ ]:
# Cell 9: Load LLM
def load_llm_model(model_name):
    """Load the LLM model for query generation."""
    print(f"Loading LLM model: {model_name}")
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )
        print(f"LLM model loaded successfully!")
        return tokenizer, model
    except Exception as e:
        print(f"Error loading LLM model: {e}")
        return None, None

# Load LLM model
tokenizer, llm_model = load_llm_model(LLM_MODEL)

In [ ]:
# Cell 10:Generate Query

def create_query_prompt(unique_terms, topic):
    """Create a prompt template for query generation."""

    prompt = f"""
    You are an AI assistant that helps people remember specific items they’ve forgotten the name of — like a movie, show, or celebrity. These are called “tip-of-the-tongue” (ToT) queries.

    Given some key concepts from a document, generate a long, natural-sounding search query that might be asked by a human trying to recall this item.
    The query should include uncertainty, vague guesses, and even possibly incorrect or fuzzy recollections.
    Avoid giving exact names or made-up facts.
    Think step by step and talk like you're trying to remember something from memory.

    Example 1:
    **Keywords**: empire, space, father, twist, 1980s  
    **Query**: What’s that old sci-fi movie from the 80s where there’s a big twist about a guy being the father of someone? It’s part of a space saga, maybe something about an empire?

    Example 2:
    **Keywords**: thriller, dance, zombies, singer  
    **Query**: What’s that Halloween music video where a famous singer turns into a zombie and dances with others? I think it was from the 80s — really popular.

    Now try this one:
    **Keywords**: {', '.join(unique_terms)}  
    **Query**:
    """
    return prompt

def generate_query_with_llm(prompt, tokenizer, model, max_length=100):
    """Generate a query using the LLM."""
    try:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_length,
                temperature=0.5,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.2,
                eos_token_id=tokenizer.eos_token_id
            )
        
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Extract only the generated part (after the prompt)
        query = generated_text[len(prompt):].strip()
        
        return query
    except Exception as e:
        print(f"Error generating query: {e}")
        return None


if tokenizer is not None and llm_model is not None:
    # Create prompt
    prompt = create_query_prompt(unique_terms, TOPIC)
    print(f"\nGenerated prompt:")
    print(prompt)
    
    # Generate query
    print(f"\nGenerating query with LLM...")
    generated_query = generate_query_with_llm(prompt, tokenizer, llm_model)
    
    if generated_query:
        print(f"\nGenerated Query: {generated_query}")
        print(f"\nTarget Document Title: {target_content['title']}")
    else:
        print("Failed to generate query")
else:
    print("LLM model not available, skipping query generation")

Loading LLM model: HuggingFaceH4/zephyr-7b-beta


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


LLM model loaded successfully!

Generated prompt:

    You are an AI assistant that helps people remember specific items they’ve forgotten the name of — like a movie, show, or celebrity. These are called “tip-of-the-tongue” (ToT) queries.

    Given some key concepts from a document, generate a long, natural-sounding search query that might be asked by a human trying to recall this item.
    The query should include uncertainty, vague guesses, and even possibly incorrect or fuzzy recollections.
    Avoid giving exact names or made-up facts.
    Think step by step and talk like you're trying to remember something from memory.

    Example 1:
    **Keywords**: empire, space, father, twist, 1980s  
    **Query**: What’s that old sci-fi movie from the 80s where there’s a big twist about a guy being the father of someone? It’s part of a space saga, maybe something about an empire?

    Example 2:
    **Keywords**: thriller, dance, zombies, singer  
    **Query**: What’s that Halloween music

In [ ]:
# Cell 10: Summary and Results

print("=" * 80)
print("QUERY GENERATION PIPELINE SUMMARY")
print("=" * 80)

print(f"\nTopic: {TOPIC}")
print(f"Target Document ID: {target_id}")
print(f"Target Document Title: {target_content['title']}")
print(f"Target Document URL: {target_content['url']}")

print(f"\nTop {FAISS_TOP_K} Similar Documents:")
for doc in similar_docs:
    print(f"  {doc['rank']}. {doc['title']} (Score: {doc['score']:.4f})")

print(f"\nTarget Keywords: {target_keywords}")
print(f"Unique Terms: {unique_terms}")

if 'generated_query' in locals() and generated_query:
    print(f"\nGenerated Query: {generated_query}")

print("\n" + "=" * 80)